<a href="https://colab.research.google.com/github/bcbutler2212/Education-Policy-Research-Chatbot/blob/Sofia/Working_Llama_3_2_1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Llama-3.2-1B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Llama-3.2-1B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [2]:
from huggingface_hub import login
login(new_session=False)

In [3]:
# Use a pipeline as a high-level helper
from transformers import pipeline
import os
from google.colab import userdata

# Set the HF_TOKEN environment variable
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B")

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

In [5]:
# load files temporarily
from google.colab import files
uploaded = files.upload()

Saving pdfs_for_chatbot.zip to pdfs_for_chatbot.zip


In [6]:
# unzip
!unzip /content/pdfs_for_chatbot.zip

Archive:  /content/pdfs_for_chatbot.zip
   creating: pdfs_for_chatbot/
  inflating: pdfs_for_chatbot/AcademicStandards_Bleiberg_Formatted.pdf  
  inflating: pdfs_for_chatbot/AssistantPrincipals_Goldring_Formatted.pdf  
  inflating: pdfs_for_chatbot/BartanenRogers_AdministratorMobility_Formatted.pdf  
  inflating: pdfs_for_chatbot/BroaderECEBenefits_Gibbs_Formatted.pdf  
  inflating: pdfs_for_chatbot/Charters_Harris_Formatted.pdf  
  inflating: pdfs_for_chatbot/Choice_Carlson_Formatted.pdf  
  inflating: pdfs_for_chatbot/CollegeAccess_Meyer_etal_Formatted.pdf  
  inflating: pdfs_for_chatbot/CollegeAdmissions_Klasik_Formatted.pdf  
  inflating: pdfs_for_chatbot/CollegePricing_CookTurner_Formatted.pdf  
  inflating: pdfs_for_chatbot/CollegeScholarships_Gurantz_Formatted.pdf  
  inflating: pdfs_for_chatbot/COVIDHigherEducation_ActonCook_Formatted.pdf  
  inflating: pdfs_for_chatbot/Covid_Zamarro_Formatted.pdf  
  inflating: pdfs_for_chatbot/Curriculum_Steiner_Formatted.pdf  
  inflating: p

In [7]:
# Install dependencies
!pip install langchain langchain-community pypdf chromadb sentence-transformers transformers

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.5 MB/s eta 0:0

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
import os
os.listdir("/content")

['.config', 'pdfs_for_chatbot.zip', 'pdfs_for_chatbot', 'sample_data']

In [10]:
# Load all pdfs

pdf_folder = "/content/pdfs_for_chatbot"

docs = []
for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        docs.extend(loader.load())

print(f"Loaded {len(docs)} PDF pages")

Loaded 815 PDF pages


In [11]:

# Split long text into smaller chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)

print(f"Created {len(split_docs)} text chunks")

Created 3765 text chunks


In [12]:
# wrap
from langchain_community.llms import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

/tmp/ipython-input-2288672069.py:3: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [13]:
# embedding model
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipython-input-3310970542.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
# Create a persistent Chroma DB
vectordb = Chroma.from_documents(split_docs, embedding=embedding_model, persist_directory="chroma_db")


In [15]:
vectordb.persist()

/tmp/ipython-input-3711397106.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


In [16]:
# If chunks print then working

results = vectordb.similarity_search("What is this document about?", k=2)
for i, r in enumerate(results, 1):
    print(f"\nResult {i}:\n{r.page_content[:500]}...")


Result 1:
bring attention to an issue, expand a person’s underȉanding of an area of policy or praǻice, or 
Ǻange the way in whiǺ someone looks at a problem.  
 
While ȉudies provide empirical support for conceptual use, measuring this type of use is 
more Ǻallenging. Moȉ researǺ to date involves self-reports through interviews or surveys. 
Furthermore, it does not examine the ways in whiǺ conceptual use may precede or be 
embedded in inȉrumental or symbolic use of researǺ evidence, for example, when 
rese...

Result 2:
Pg. 6 
 
 
 
 
Curriculum Policy  |   David M. Steiner 
 
among elementary -school children. Eighth -grade results have not shown the same 
jumps.31 One possible explanation is that while policymakers have focused on phonics 
instruction, one key element of the science of reading, namely, the building of background 
knowledge, has not , until very recently , been at the core of the reforms in literacy 
instructional materials.  
The idea is simple but central: if studen

In [17]:
# make retriever --> return 3 chunks? change this?
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [18]:
from langchain.chains import RetrievalQA

# connect llama with chroma
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)

In [19]:
# test it!
result = qa_chain.invoke({"query": "Summarize the key findings from the PDFs"})
print(result["result"])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

On the other hand, i nterim assessments have been helpful as outcome measure s to study 
the effects of a particular intervention or event (e.g., COVID -19 disruptions) on student 
achievement. Depending on the type of study, the ability to track student performance

for success in life. 
 
Key ﬁnding #н: SǺools could improve ȉudent aǺievement by seleǻing teaǺers—hiring, 
retaining, tenuring, and ﬁring—based on their performance scores. 
 
Policies that tie seleǻion decisions—hiring, ﬁring, tenure—direǻly to a teaǺer’s measured 
performance are ȉill uncommon, despite the growing availability of evaluation measures. 
Notable exceptions include the IMPACT program in Washington, DC, described above, and 
several ȉates and diȉriǻs conditioning tenure on measured performance, described below. 
 
Given the scarcity of exiȉing poli

In [20]:
question = "What are school distruptions?"
result = qa_chain.invoke({"query": question})
print(result["result"])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

about school funding, concerns with financing mechanisms, or the role and impact of the 
courts. Nonetheless, the continuing application to the courts for changes in the structure 
of state school finance arrangements indicate s a degree of ongoing dissatisfaction with 
school policymaking.

school finance policy and decisions. 
Decision makers in the executive and legislative branches of state and federal 
governments, along with those in local school districts , have developed a variety of 
approaches for ensuring that schools have the resources necessary to do their job. The 
funding patterns that have resulted continue to be questioned on the grounds of 
inequitable opportunities across districts. Additionally, the results, as measured in terms 
of student outcomes , have not constituted the outcomes that are desired. 43